In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # lee el archivo .env

codigos_fred = {
    "fed_funds": "FEDFUNDS",
    "ust10":     "GS10",
    "cpi_us":    "CPIAUCSL",
    "VIX":     "VIXCLS",
    "dollar_index": "DTWEXBGS",
    "cobre_fred": "PCOPPUSDM"
}

url_fred = "https://api.stlouisfed.org/fred/series/observations"
series_fred = {}
for nombre, codigo in codigos_fred.items():
    params = {
        "series_id": codigo,                    # ← el código cambia en cada vuelta
        "api_key": os.getenv("FRED_API_KEY"),
        "file_type": "json",
        "observation_start": "2010-01-01",
    }
    r = requests.get(url_fred, params=params, timeout=30)
    r.raise_for_status()
    fred = pd.DataFrame(r.json()["observations"])
    fred["date"] = pd.to_datetime(fred["date"])
    fred["value"] = pd.to_numeric(fred["value"], errors="coerce")  # convierte texto a número
    series_fred[nombre] = fred.set_index("date")["value"]

base_f = pd.DataFrame(series_fred)   # una columna por variable, una fila por fecha
print(base.head())

            fed_funds  ust10  cpi_us    VIX  dollar_index  cobre_fred
date                                                                 
2026-10-05        NaN    NaN     NaN  15.52           NaN         NaN
2026-10-06        NaN    NaN     NaN  15.01           NaN         NaN
2026-10-07        NaN    NaN     NaN  15.08           NaN         NaN
2026-10-08        NaN    NaN     NaN  15.41           NaN         NaN
2026-10-09        NaN    NaN     NaN  14.84           NaN         NaN


In [ ]:
print(base_f.describe())
print(base_f.isna().sum())  # cuántos valores faltan por columna

        fed_funds       ust10      cpi_us          VIX  dollar_index  \
count  201.000000  201.000000  199.000000  4251.000000   4172.000000   
mean     1.515672    2.695274  260.983357    18.396104    109.318201   
std      1.832159    1.051403   34.376062     6.740273     11.999361   
min      0.050000    0.620000  217.199000     9.140000     85.469200   
25%      0.100000    1.930000  235.315000    13.880000     94.447675   
50%      0.370000    2.540000  250.227000    16.700000    113.269450   
75%      2.410000    3.570000  293.105500    20.950000    119.123475   
max      5.330000    4.990000  334.131000    82.690000    130.041300   

         cobre_fred  
count    199.000000  
mean    7668.433753  
std     1877.610200  
min     4471.787500  
25%     6235.168478  
50%     7428.289474  
75%     8984.682777  
max    13552.040909  
fed_funds       4232
ust10           4232
cpi_us          4234
VIX              182
dollar_index     261
cobre_fred      4234
dtype: int64


In [10]:
codigos_bcch = {
    "dolar_observado": "F073.TCO.PRE.Z.D",
    "cuenta_corriente": "F068.A.FLU.Z.0.S.N.Z.Z.Z.Z.6.0.T",
    "tpm": "F022.TPM.TIN.D001.NO.Z.D",
    "ipc": "F074.IPC.IND.Z.EP23.C.M",
    "imacec": "F032.IMC.IND.Z.Z.EP18.Z.Z.0.M",
    "cobre_refinado": "F019.PPB.PRE.40.M",
    "exportaciones_cobre_dolares": "F068.B1.FLU.A1.0.C.N.Z.Z.Z.Z.6.0.M",
    "exportaciones_cobre_toneladas": "F034.ECUM.FLU.COCHILCO.Z.0.M",
    "total_ingresos": "F051.BT02.FLU.C.D.CLP.M",
    "total_gastos": "F051.BT03.FLU.C.D.CLP.M",
}

url_bcch = "https://si3.bcentral.cl/SieteRestWS/SieteRestWS.ashx"
series_bcch = {}

for nombre, codigo in codigos_bcch.items():
    params = {
        "user": os.getenv("BCCH_USER"),
        "pass": os.getenv("BCCH_PASS"),
        "firstdate": "2010-01-01",
        "timeseries": codigo,
        "function": "GetSeries",          # obligatorio en el BCCh
    }
    r = requests.get(url_bcch, params=params, timeout=30)
    r.raise_for_status()
    datos = r.json()

    # si el BCCh responde con error, lo muestra y pasa a la siguiente serie
    if datos["Codigo"] != 0:
        print(f"Error en {nombre}: {datos['Descripcion']}")
        continue

    bcch = pd.DataFrame(datos["Series"]["Obs"])
    bcch["fecha"] = pd.to_datetime(bcch["indexDateString"], format="%d-%m-%Y")
    bcch["valor"] = pd.to_numeric(bcch["value"], errors="coerce")
    series_bcch[nombre] = bcch.set_index("fecha")["valor"]
    print(f"OK {nombre}: {len(bcch)} observaciones")

base_bcch = pd.DataFrame(series_bcch)
print(base_bcch.tail())

OK dolar_observado: 6127 observaciones
OK cuenta_corriente: 66 observaciones
OK tpm: 6127 observaciones
OK ipc: 201 observaciones
OK imacec: 200 observaciones
OK cobre_refinado: 201 observaciones
OK exportaciones_cobre_dolares: 201 observaciones
OK exportaciones_cobre_toneladas: 199 observaciones
OK total_ingresos: 198 observaciones
OK total_gastos: 198 observaciones
            dolar_observado  cuenta_corriente  tpm  ipc  imacec  \
fecha                                                             
2026-10-09           981.61               NaN  4.5  NaN     NaN   
2026-10-10              NaN               NaN  NaN  NaN     NaN   
2026-10-11              NaN               NaN  NaN  NaN     NaN   
2026-10-12              NaN               NaN  NaN  NaN     NaN   
2026-10-13           978.61               NaN  4.5  NaN     NaN   

            cobre_refinado  exportaciones_cobre_dolares  \
fecha                                                     
2026-10-09             NaN                